In [ ]:
import torch 
import torch.nn as nn

class GQA(nn.Module):

    def __init__(self, embed_dim, num_heads, kv_heads):
        super().__init__()

        assert num_heads % kv_heads == 0
        assert embed_dim % num_heads == 0

        self.num_heads = num_heads
        self.kv_heads = kv_heads
        self.heads_per_kv_head = num_heads // kv_heads
        self.head_dim = embed_dim // num_heads 

        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, kv_heads * self.head_dim)
        self.v = nn.Linear(embed_dim, kv_heads * self.head_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, causal = False):

        b, t, e = x.shape

        q = self.q(x).reshape(b, t, self.num_heads, -1).transpose(1,2)
        k = self.k(x).reshape(b, t, self.kv_heads, -1).transpose(1,2)
        v = self.v(x).reshape(b, t, self.kv_heads, -1).transpose(1,2)

        # (b, heads, t, head_dim)
        k = torch.repeat_interleave(k, self.heads_per_kv_head, dim = 1)
        v = torch.repeat_interleave(v, self.heads_per_kv_head, dim = 1)

        scores = q @ k.transpose(-2, -1) / (self.head_dim ** 0.5)

        if causal:
            mask = torch.tril(torch.ones((t,t), dtype = torch.bool))
            scores = scores.masked_fill(~mask, float("-inf"))

        p = torch.softmax(scores, dim = -1)
        output = p @ v
        return self.proj(output.transpose(1,2).reshape(b, t, -1))